# Data Cleaning

In [72]:
import pandas as pd
import numpy as np

## Load Dataset

In [73]:
original_data = pd.read_csv('data/raw.csv', low_memory=False)
data = original_data.copy()

## Remove Duplicate Rows

In [74]:
duplicates = data[data.duplicated(keep=False)]
duplicates.sort_values("id").head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
676,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
1465,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
14012,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",http://www.dealthemovie.com/,11115,tt0446676,en,Deal,As an ex-gambler teaches a hot-shot college ki...,...,2008-01-29,0.0,85.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Deal,False,5.2,22.0
24844,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",http://www.dealthemovie.com/,11115,tt0446676,en,Deal,As an ex-gambler teaches a hot-shot college ki...,...,2008-01-29,0.0,85.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Deal,False,5.2,22.0
19890,False,NaN,0,"[{'id': 14, 'name': 'Fantasy'}, {'id': 18, 'na...",NaN,119916,tt0080000,en,The Tempest,"Prospero, the true Duke of Milan is now living...",...,1980-02-27,0.0,123.0,[],Released,NaN,The Tempest,False,0.0,0.0


In [75]:
data = data.drop_duplicates().reset_index(drop=True)
print("duplicate data delete!")
print(f"Duplicate rows after cleaning: {data.duplicated().sum()}")

duplicate data delete!
Duplicate rows after cleaning: 0


## Fixed Data Type

### Data Type Issues (9 Columns Need Conversion)

| Column | Current → Expected |
|--------|-------------------|
| `adult`, `video` | object → bool |
| `budget`, `id`, `popularity` | object → int64/float64 |
| `revenue`, `runtime`, `vote_count` | float64 → int64 |
| `release_date` | object → datetime64[ns] |

In [76]:
numeric_columns = [
    "budget",
    "id",
    "popularity",
    "revenue",
    "runtime",
    "vote_average",
    "vote_count"
]
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    print(f"type of {col} changed to {data[col].dtype}")

type of budget changed to float64
type of id changed to float64
type of popularity changed to float64
type of revenue changed to float64
type of runtime changed to float64
type of vote_average changed to float64
type of vote_count changed to float64


In [77]:
data["release_date"] = pd.to_datetime(data["release_date"], errors="coerce")
print("type of {} changed to {}".format("release_date", data["release_date"].dtype))

type of release_date changed to datetime64[ns]


### Clean Boolean Columns

3 rows in `adult` column contain text/overview data (likely import shift)

In [78]:
invalid_data = data[~data["adult"].isin(['True', 'False'])]
invalid_data

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19725,- Written by Ørnås,0.065736,NaN,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",NaN,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29491,Rune Balot goes to a casino connected to the ...,1.931659,NaN,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",NaN,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35575,Avalanche Sharks tells the story of a bikini ...,2.185485,NaN,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",NaN,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
# Remove corrupted rows caused by CSV parsing issues
data = data[data["adult"].isin(['True', 'False'])]
print(f"Invalid rows in 'adult' column: {len(data[~data['adult'].isin(['True', 'False'])])}")

Invalid rows in 'adult' column: 0


In [80]:
boolean_col = ["video", "adult"]
for col in boolean_col:
    data[col] = data[col].map({'True': True, 'False': False}).astype('boolean')
    print(f"type of {col} changed to {data[col].dtype}")

type of video changed to boolean
type of adult changed to boolean


### Cleaning Summary

- Removed 3 corrupted rows caused by CSV parsing issues.
- Converted `adult` and `video` to Boolean dtype.
- Dataset is now consistent for Boolean features.

## Parse JSON Columns

In [ ]:
# Check the structure of a sample rows
# Define JSON columns and their corresponding parsed column names
json_columns = ['belongs_to_collection', 'genres', 'production_companies'
                , 'production_countries', 'spoken_languages']

parse_columns = ['collection', 'genre_names', 'company_names', 
                 'country_names', 'language_names']

# Create a DataFrame to inspect sample data types 
dtype_sample = pd.DataFrame({
    'Column': json_columns,
    'Sample Value':  [data[col].iloc[0] for col in json_columns],
    'Sample_Dtype': [type(data[col].iloc[0]) for col in json_columns],
})

dtype_sample

,Column,Sample Value,Sample_Dtype,Sample[0]_Dtype
0,belongs_to_collection,"{'id': 10194, 'name': 'Toy Story Collection', ...",<class 'str'>,<class 'str'>
1,genres,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",<class 'str'>,<class 'str'>
2,production_companies,"[{'name': 'Pixar Animation Studios', 'id': 3}]",<class 'str'>,<class 'str'>
3,production_countries,"[{'iso_3166_1': 'US', 'name': 'United States o...",<class 'str'>,<class 'str'>
4,spoken_languages,"[{'iso_639_1': 'en', 'name': 'English'}]",<class 'str'>,<class 'str'>


In [82]:
# Convert string representations to actual Python objects

import ast

def parse_json_column(series):
    """Safely convert string representations of JSON to Python objects."""
    return series.apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# Apply parsing to all JSON columns
for col in json_columns:
    data[col] = parse_json_column(data[col])

In [83]:
# Check data types after conversion

final_dtype = pd.DataFrame({
    'Column': json_columns,
    'Sample_Dtype': [type(data[col].iloc[0]) for col in json_columns],
    'Sample[0]_Dtype': [ type([data[col].iloc[0]][0]) for col in json_columns]
})

final_dtype

,Column,Sample_Dtype,Sample[0]_Dtype
0,belongs_to_collection,<class 'dict'>,<class 'dict'>
1,genres,<class 'list'>,<class 'list'>
2,production_companies,<class 'list'>,<class 'list'>
3,production_countries,<class 'list'>,<class 'list'>
4,spoken_languages,<class 'list'>,<class 'list'>


In [87]:
# Extract names from JSON columns (list of dicts or single dict)

def extract_names(items):
    """
    Extract 'name' values from a list of dictionaries or a single dictionary.
    """
    # Handle None or NaN values
    # Check type first to avoid pandas ValueError
    if items is None:
        return []
    if isinstance(items, float) and pd.isna(items):
        return []
    
    # Input is a list of dictionaries
    if isinstance(items, list):
        # List comprehension with safety checks
        return [
            item.get('name', '') 
            for item in items 
            if isinstance(item, dict) and item.get('name')
        ]
    
    # Input is a single dictionary (belongs_to_collection)
    elif isinstance(items, dict):
        name = items.get('name', '')
        return [name] if name else []
    
    # Any other data type (invalid)
    return []


# Apply extraction to all columns
for col, new_col in zip(json_columns, parse_columns):
    data[new_col] = data[col].apply(extract_names)
    print(f"Created '{new_col}' from '{col}'")

Created 'collection' from 'belongs_to_collection'
Created 'genre_names' from 'genres'
Created 'company_names' from 'production_companies'
Created 'country_names' from 'production_countries'
Created 'language_names' from 'spoken_languages'


In [88]:
# Verify the final results
final_sample = pd.DataFrame({
    'Column': parse_columns,
    'Final Sample':  [data[col].iloc[0] for col in parse_columns]})

final_sample

,Column,Final Sample
0,collection,[Toy Story Collection]
1,genre_names,"[Animation, Comedy, Family]"
2,company_names,[Pixar Animation Studios]
3,country_names,[United States of America]
4,language_names,[English]


## Clean Numeric Columns

In [90]:
data[numeric_columns].describe()

,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4.544600e+04,45446.000000,45443.000000,4.544300e+04,45186.000000,45443.000000,45443.000000
mean,4.226138e+06,108359.140782,2.921510,1.121350e+07,94.126234,5.618311,109.922210
std,1.742720e+07,112470.216845,6.006106,6.434392e+07,38.412464,1.924092,491.398368
min,0.000000e+00,2.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,0.000000e+00,26450.250000,0.386012,0.000000e+00,85.000000,5.000000,3.000000
50%,0.000000e+00,59995.500000,1.127613,0.000000e+00,95.000000,6.000000,10.000000
75%,0.000000e+00,157332.000000,3.679023,0.000000e+00,107.000000,6.800000,34.000000
max,3.800000e+08,469172.000000,547.488298,2.787965e+09,1256.000000,10.000000,14075.000000


In [91]:
data[numeric_columns].isna().sum()

budget            0
id                0
popularity        3
revenue           3
runtime         260
vote_average      3
vote_count        3
dtype: int64